# Pig Runner + r2dreamer, on Colab's free GPU

Trains DreamerV3 (via [r2dreamer](https://github.com/NM512/r2dreamer)) on the
Pig Runner environment, then renders a rollout of the trained agent to a GIF
and shows it right here — no terminal, no window, nothing local required.

**Before running:** Runtime -> Change runtime type -> GPU.

Replace `PIG_RUNNER_REPO` below with this repo's clone URL.

In [ ]:
PIG_RUNNER_REPO = "https://github.com/Sweeyya/pig-runner.git"
R2DREAMER_REPO = "https://github.com/NM512/r2dreamer.git"
LOGDIR = "/content/pigrunner_run"
TRAIN_STEPS = 5e4   # small demo run -- bump way up for anything close to the
                    # ~17-19 score the scripted expert gets

In [ ]:
!git clone -q $PIG_RUNNER_REPO /content/pig-runner
!git clone -q $R2DREAMER_REPO /content/r2dreamer
%cd /content/r2dreamer
!pip install -q -r requirements.txt
!pip install -q -r /content/pig-runner/requirements.txt

## Wire the environment in

Copies the pure logic files into r2dreamer's `envs/`, drops in the config,
and patches `make_env()` with the `pigrunner` branch -- the same three steps
from the main README, done here instead of by hand.

In [ ]:
init_path = envs_dir / "__init__.py"
init_src = init_path.read_text()

branch_lines = (src / "integration/make_env_branch.py").read_text().splitlines()
branch_code = "\n".join(l for l in branch_lines if not l.strip().startswith("#"))

marker = '    else:\n        raise NotImplementedError(suite)'
assert marker in init_src, "envs/__init__.py layout changed upstream -- patch make_env() by hand"
init_src = init_src.replace(marker, branch_code.rstrip("\n") + "\n" + marker)
init_path.write_text(init_src)
print("Patched envs/__init__.py with the pigrunner branch.")


## Train

`time_limit: 1000` in `pigrunner.yaml` is r2dreamer's own `TimeLimit`
wrapper doing the 1000-step cap -- this env only ever reports real death.

In [ ]:
!python train.py env=pigrunner logdir=$LOGDIR env.steps=$TRAIN_STEPS device=cuda:0

## Watch the trained agent

Loads `latest.pt`, rolls out one episode with the environment's own headless
renderer (`render_rgb` -- the same one `play.py` uses locally, no display
server needed), and writes a GIF.

In [ ]:
import sys
sys.path.append("/content/r2dreamer")
sys.path.append("/content/pig-runner")  # so `import render` resolves, even
                                         # though render.py isn't copied into envs/
import torch
import hydra
from dreamer import Dreamer
from envs import wrappers
from envs.pigrunner import PigRunnerEnv

device = "cuda:0" if torch.cuda.is_available() else "cpu"
raw_env = PigRunnerEnv(task="v0", seed=0)
env = wrappers.OneHotAction(raw_env)

# Rebuild the exact config train.py used (env=pigrunner), the same way its
# own @hydra.main decorator does, so the model architecture matches the
# checkpoint's weights.
with hydra.initialize(config_path="configs", version_base=None):
    config = hydra.compose(config_name="configs", overrides=["env=pigrunner"])

agent = Dreamer(config.model, env.observation_space, env.action_space)
ckpt = torch.load(f"{LOGDIR}/latest.pt", map_location=device)
agent.load_state_dict(ckpt["agent_state_dict"])
agent.to(device).eval()
print("Loaded", f"{LOGDIR}/latest.pt")


In [ ]:
obs = env.reset()
state = agent.get_initial_state(1)
frames, score = [], 0
for _ in range(1000):
    batched = {
        k: torch.as_tensor(v, device=device, dtype=torch.float32)[None]
        if k == "state" else torch.as_tensor([v], device=device)
        for k, v in obs.items()
    }
    action, state = agent.act(batched, state, eval=True)
    obs, reward, done, info = env.step(action[0].cpu().numpy())
    score += reward
    frames.append(raw_env.render())
    if done:
        break
print(f"episode score: {score}, length: {len(frames)}")


In [ ]:
import imageio
from IPython.display import Image, display

imageio.mimsave("/content/rollout.gif", frames, fps=50, loop=0)
display(Image(filename="/content/rollout.gif"))

## Training curve

r2dreamer logs scalars with `tools.Logger` under `logdir` (TensorBoard
event files). Point TensorBoard at it directly rather than re-parsing logs
by hand:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $LOGDIR

## Watch the world model dream

Optional, and separate from everything above. DreamerV3 trains its policy
almost entirely on trajectories it *imagines* inside its own learned world
model, without touching the real environment. This section makes that
imagination visible: it gives the model a few real steps of context, then
lets it roll forward purely from its own predictions -- using the actions
that were *actually* taken next -- and renders both, side by side. Where
they diverge is where the model's understanding breaks down.

**This needs a separate training run.** r2dreamer's default mode
(`rep_loss: r2dreamer`) trains without a decoder at all -- that's where its
speed advantage over classic DreamerV3 comes from -- so there is nothing to
decode imagined states back into numbers with. `pigrunner_dream.yaml`
switches to the classic reconstruction objective (`rep_loss: dreamer`)
instead, trading that speed for the ability to do this at all.

In [ ]:
shutil.copy(src / "integration/pigrunner_dream.yaml",
            "/content/r2dreamer/configs/model/pigrunner_dream.yaml")

DREAM_LOGDIR = "/content/pigrunner_dream_run"
DREAM_TRAIN_STEPS = 5e4  # same caveat as TRAIN_STEPS above -- bump way up for a real result

In [ ]:
!python train.py env=pigrunner model=pigrunner_dream logdir=$DREAM_LOGDIR env.steps=$DREAM_TRAIN_STEPS device=cuda:0

In [ ]:
import hydra
import torch
from dreamer import Dreamer
from envs import wrappers
from envs.pigrunner import PigRunnerEnv

with hydra.initialize(config_path="configs", version_base=None):
    dream_config = hydra.compose(config_name="configs",
                                  overrides=["env=pigrunner", "model=pigrunner_dream"])

dream_raw_env = PigRunnerEnv(task="v0", seed=1)
dream_env = wrappers.OneHotAction(dream_raw_env)

dream_agent = Dreamer(dream_config.model, dream_env.observation_space, dream_env.action_space)
dream_ckpt = torch.load(f"{DREAM_LOGDIR}/latest.pt", map_location=device)
dream_agent.load_state_dict(dream_ckpt["agent_state_dict"])
dream_agent.to(device).eval()
print("Loaded", f"{DREAM_LOGDIR}/latest.pt")

Step through some real gameplay, logging every observation and action --
then replay that exact action sequence through the world model twice: once
as a posterior pass (grounded in the real observations, for context), once
as pure imagination (`rssm.imagine_with_action`, no observations at all).

In [ ]:
CONTEXT_STEPS = 5
IMAGINE_STEPS = 30

obs = dream_env.reset()
state = dream_agent.get_initial_state(1)
real_frames, logged_states, logged_actions, logged_is_first = [], [], [], []

for _ in range(CONTEXT_STEPS + IMAGINE_STEPS):
    batched = {
        k: torch.as_tensor(v, device=device, dtype=torch.float32)[None]
        if k == "state" else torch.as_tensor([v], device=device)
        for k, v in obs.items()
    }
    action, state = dream_agent.act(batched, state, eval=True)
    logged_states.append(obs["state"])
    logged_is_first.append(obs["is_first"])
    logged_actions.append(action[0].cpu())
    obs, reward, done, info = dream_env.step(action[0].cpu().numpy())
    real_frames.append(dream_raw_env.render())
    if done:
        break

T = len(logged_states)
print(f"logged {T} real steps ({CONTEXT_STEPS} context + up to {IMAGINE_STEPS} to compare against)")

In [ ]:
state_seq = torch.as_tensor(logged_states, dtype=torch.float32, device=device)[None]   # (1, T, 9)
is_first_seq = torch.as_tensor(logged_is_first, device=device)[None]                    # (1, T)
action_seq = torch.stack(logged_actions, dim=0)[None].to(device)                        # (1, T, A)
data = {"state": state_seq, "is_first": is_first_seq}

p_data = dream_agent.preprocess(data)
embed = dream_agent.encoder(p_data)

ctx = min(CONTEXT_STEPS, T)
initial = dream_agent.rssm.initial(1)
post_stoch, post_deter, _ = dream_agent.rssm.observe(
    embed[:, :ctx], action_seq[:, :ctx], initial, is_first_seq[:, :ctx]
)
init_stoch, init_deter = post_stoch[:, -1], post_deter[:, -1]

recon = dream_agent.decoder(post_stoch, post_deter)["state"].mode()[0]  # (ctx, 9)
if ctx < T:
    prior_stoch, prior_deter = dream_agent.rssm.imagine_with_action(
        init_stoch, init_deter, action_seq[:, ctx:]
    )
    imagined = dream_agent.decoder(prior_stoch, prior_deter)["state"].mode()[0]  # (T-ctx, 9)
    predicted = torch.cat([recon, imagined], dim=0)
else:
    predicted = recon
predicted_states = predicted.cpu().numpy()  # (T, 9) -- decoded belief at every step

Render both tracks with the same renderer used everywhere else in this repo
-- `draw_world` for the real side, `draw_dream` (new, in `render.py`) for the
predicted side, which draws only what the model actually predicted: a pig at
the predicted height, and a generic translucent hazard band at the predicted
distance and height range -- never a specific sprite, since *which* hazard
it is was never one of the numbers it predicted.

In [ ]:
import numpy as np
import pygame
import world as W
from render import draw_dream, draw_dream_label

pygame.font.init()
dream_font = pygame.font.SysFont("menlo,monaco,consolas,monospace", 14, bold=True)
frames = []
for t in range(T):
    dream_surf = pygame.Surface((W.NATIVE_W, W.NATIVE_H))
    draw_dream(dream_surf, predicted_states[t])
    draw_dream_label(dream_surf, dream_font, "DREAM" if t >= ctx else "DREAM (context)")
    dream_frame = np.transpose(pygame.surfarray.array3d(dream_surf), (1, 0, 2))

    real_frame = real_frames[t]
    combo = np.zeros((real_frame.shape[0] * 2 + 4, real_frame.shape[1], 3), dtype=np.uint8)
    combo[:real_frame.shape[0]] = real_frame
    combo[real_frame.shape[0] + 4:] = dream_frame
    frames.append(combo)

imageio.mimsave("/content/dream.gif", frames, fps=50, loop=0)
display(Image(filename="/content/dream.gif"))